# Data Harmonization for Multi-Omics

**Tier 3 — Applied Bioinformatics | Module 27 · Notebook 1**

*Prerequisites: Module 03 (RNA-seq), Module 07 (Machine Learning), Module 18 (Proteomics)*

---

**By the end of this notebook you will be able to:**
1. Describe the challenges of integrating heterogeneous omics data
2. Apply sample-centric and feature-centric scaling strategies
3. Handle missing data across omics layers (imputation strategies)
4. Detect and visualize batch effects between omics platforms
5. Prepare a harmonized multi-omics matrix for downstream integration


**Key resources:**
- [MOFA2 documentation](https://biofam.github.io/MOFA2/)
- [mixOmics documentation](http://mixomics.org/)
- [Galaxy Training — Multi-Omics](https://training.galaxyproject.org/)

---[< Previous: QIIME2 16S Amplicon Workflow](../26_Metagenomics_Shotgun/04_qiime2_16s.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: MOFA2: Multi-Omics Factor Analysis >](02_mofa2.ipynb)---

## 1. The Multi-Omics Data Landscape

Multi-omics integration combines measurements from different molecular layers to gain a holistic view of biological systems.

### Data modalities and their properties

| Omics layer | Measures | Typical features | Data type |
|---|---|---|---|
| **Genomics** (WGS/WES) | DNA variants, CNVs | ~4M SNPs | Binary/integer |
| **Transcriptomics** (RNA-seq) | mRNA abundance | ~20,000 genes | Continuous (counts) |
| **Proteomics** (LC-MS) | Protein abundance | 3,000-10,000 proteins | Continuous, many missing |
| **Metabolomics** | Metabolite levels | 500-5,000 metabolites | Continuous |
| **Epigenomics** (ATAC/ChIP) | Chromatin accessibility | ~100k-1M peaks | Binary/continuous |
| **Methylomics** (RRBS/WGBS) | DNA methylation | ~500k-28M CpGs | Beta (0-1) |

### Integration strategies

**Concatenation (early integration)**: merge all matrices horizontally → dimension reduction
**Meta-analysis (late integration)**: analyze each omics separately → combine results
**Factor analysis (intermediate)**: MOFA2, iCluster — learn shared latent factors
**Supervised (DIABLO)**: use sample labels to guide integration

### Key challenges
- Vastly different feature dimensionalities (p >> n for each layer)
- Missing data in proteomics (~20-40% MNAR)
- Batch effects from different processing dates/platforms
- Scale differences: log-counts vs beta values vs raw intensities


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer

np.random.seed(42)

# ----- Generate synthetic multi-omics dataset -----
# 60 samples: 20 normal, 20 tumor type A, 20 tumor type B
n_samples = 60
n_rna = 2000   # RNA-seq features (genes)
n_prot = 300   # proteomics features
n_meth = 500   # methylation features (CpG sites)

sample_ids = [f'S{i:03d}' for i in range(n_samples)]
sample_type = ['Normal']*20 + ['TumorA']*20 + ['TumorB']*20

# Two batches (different sequencing dates)
batch = ['Batch1']*30 + ['Batch2']*30

# Simulate RNA-seq (log-normalized counts)
rna_matrix = np.random.randn(n_samples, n_rna)
# Add true biological signal
for i in range(20, 40):   # TumorA signal
    rna_matrix[i, :200] += 2.0
for i in range(40, 60):   # TumorB signal
    rna_matrix[i, 200:400] += 2.0
# Add batch effect
for i in range(30):  # Batch 2
    rna_matrix[i] += np.random.normal(2.0, 0.5, n_rna)

# Simulate proteomics (MS intensities, log2-transformed)
prot_matrix = np.random.randn(n_samples, n_prot)
for i in range(20, 40):
    prot_matrix[i, :50] += 1.5
for i in range(40, 60):
    prot_matrix[i, 50:100] += 1.5
prot_matrix[:30] += np.random.normal(1.5, 0.3)  # batch effect

# Simulate methylation (beta values 0-1)
meth_matrix = np.random.beta(2, 5, (n_samples, n_meth))
for i in range(20, 40):
    meth_matrix[i, :80] = np.clip(meth_matrix[i, :80] + 0.3, 0, 1)
for i in range(40, 60):
    meth_matrix[i, 80:160] = np.clip(meth_matrix[i, 80:160] + 0.3, 0, 1)

# Metadata
metadata = pd.DataFrame({
    'SampleID': sample_ids,
    'Type': sample_type,
    'Batch': batch,
    'Age': np.random.randint(30, 75, n_samples),
    'Sex': np.random.choice(['M', 'F'], n_samples)
})

print("Multi-omics dataset created:")
print(f"  RNA-seq: {rna_matrix.shape[0]} samples x {rna_matrix.shape[1]} features")
print(f"  Proteomics: {prot_matrix.shape[0]} samples x {prot_matrix.shape[1]} features")
print(f"  Methylation: {meth_matrix.shape[0]} samples x {meth_matrix.shape[1]} features")
print(f"\nMetadata:")
print(metadata.groupby(['Type', 'Batch']).size().to_string())


## 2. Normalization and Scaling

Different omics layers require different normalization strategies before integration.

### Normalization by data type

| Data type | Recommended normalization | Rationale |
|---|---|---|
| RNA-seq counts | TMM/DESeq2 VST → z-score | Library size + biological variation |
| Proteomics intensities | Median centering + z-score | Inter-sample variation |
| Methylation beta | logit transform (M-value) → z-score | Beta values are bounded 0-1 |
| Metabolomics | Probabilistic quotient + log | Dilution effects |

### Z-score vs quantile normalization

**Z-score**: preserves relative differences between features; assumes approximately normal distribution.

**Quantile normalization**: forces identical distributions across samples; removes global sample-level effects but can distort signal.

**Key principle**: normalize *within* each omics layer before integration. Applying the same normalization across different omics would not be appropriate.


In [ ]:
# ----- Normalization and scaling strategies -----

def compare_normalizations(data, title, sample_labels, n_show=20):
    """Compare z-score, min-max, and quantile normalization."""
    scalers = {
        'Raw': data,
        'Z-score': StandardScaler().fit_transform(data),
        'Min-Max': MinMaxScaler().fit_transform(data),
        'Quantile\n(normal)': QuantileTransformer(output_distribution='normal',
                                                    random_state=42).fit_transform(data),
    }
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'Normalization Comparison: {title}', fontsize=12, fontweight='bold')
    
    type_colors = {'Normal': 'green', 'TumorA': 'orange', 'TumorB': 'red'}
    colors = [type_colors[t] for t in sample_type]
    
    for j, (norm_name, norm_data) in enumerate(scalers.items()):
        # Distribution of first 5 features
        axes[0, j].boxplot([norm_data[:, k] for k in range(5)],
                           patch_artist=True,
                           boxprops=dict(facecolor='steelblue', alpha=0.5))
        axes[0, j].set_title(f'{norm_name}\n(feature distributions)')
        axes[0, j].set_xlabel('Feature index')
        axes[0, j].set_xticks(range(1, 6))
        axes[0, j].set_xticklabels([f'F{k}' for k in range(5)])
        
        # PCA plot
        pca = PCA(n_components=2)
        pcs = pca.fit_transform(norm_data)
        for s, (x, y) in enumerate(pcs):
            axes[1, j].scatter(x, y, c=type_colors[sample_type[s]], s=20, alpha=0.7)
        axes[1, j].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        axes[1, j].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        axes[1, j].set_title(f'{norm_name}\nPCA')
    
    handles = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
    axes[1, -1].legend(handles=handles, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

compare_normalizations(rna_matrix, 'RNA-seq', sample_type)
print("Key observation: Quantile normalization equalizes feature distributions")
print("but can over-correct if batch effect is confounded with biology.")


## 3. Missing Data in Multi-Omics

Missing data mechanisms affect imputation strategy:

### Missing data mechanisms
- **MCAR** (Missing Completely at Random): random technical failures → KNN or mean imputation
- **MAR** (Missing at Random): depends on observed data → KNN imputation
- **MNAR** (Missing Not at Random): missing because value is too low to detect → min/2 imputation

### Proteomics missing data
In LC-MS proteomics, missing values predominantly follow **MNAR** (low-abundance proteins below detection limit). Best practices:
- Filter proteins missing in >50% of samples
- For remaining: half-minimum imputation for MNAR; KNN for MAR
- Flag imputed values for downstream sensitivity analysis

### RNA-seq: no traditional missing values
Zeros in RNA-seq are true zeros (not detected), not missing data. Handle with count-based models (DESeq2, edgeR) that account for zero-inflation.


In [ ]:
# ----- Missing data imputation for proteomics -----

# Introduce missing values (MS data has lots of missing - ~30%)
prot_missing = prot_matrix.copy()
missing_mask = np.random.random(prot_missing.shape) < 0.25  # 25% missing
prot_missing[missing_mask] = np.nan

print(f"Missing values introduced: {missing_mask.sum()} / {missing_mask.size}")
print(f"Missing rate: {missing_mask.mean()*100:.1f}%")

# Missing per feature
miss_per_feat = np.isnan(prot_missing).mean(axis=0)
miss_per_sample = np.isnan(prot_missing).mean(axis=1)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Missing Data Analysis and Imputation Strategies', fontsize=12, fontweight='bold')

# 1. Missing data heatmap (subset)
sns.heatmap(np.isnan(prot_missing[:, :50]).T, ax=axes[0,0],
            cmap='RdYlGn_r', xticklabels=False, yticklabels=False,
            cbar_kws={'label': 'Missing (1=yes)'})
axes[0,0].set_xlabel('Samples'); axes[0,0].set_ylabel('Features (first 50)')
axes[0,0].set_title('Missing Data Pattern')

# 2. Missing rate distribution per feature
axes[0,1].hist(miss_per_feat * 100, bins=25, color='steelblue', alpha=0.8)
axes[0,1].axvline(50, color='red', linestyle='--', label='50% threshold')
axes[0,1].set_xlabel('Missing rate per feature (%)')
axes[0,1].set_ylabel('Count'); axes[0,1].set_title('Feature Missing Rate Distribution')
axes[0,1].legend()

# 3. Missing rate per sample
axes[0,2].bar(range(n_samples), miss_per_sample * 100, color='salmon', alpha=0.8)
axes[0,2].set_xlabel('Sample index'); axes[0,2].set_ylabel('Missing rate (%)')
axes[0,2].set_title('Sample Missing Rate')
axes[0,2].axhline(40, color='red', linestyle='--', label='40% threshold')
axes[0,2].legend()

# 4-6. Imputation methods comparison
methods = {'KNN\n(k=5)': KNNImputer(n_neighbors=5).fit_transform(prot_missing),
           'Mean': np.where(np.isnan(prot_missing),
                            np.nanmean(prot_missing, axis=0), prot_missing),
           'Min/2\n(MNAR)': np.where(np.isnan(prot_missing),
                                      np.nanmin(prot_missing, axis=0) / 2, prot_missing)}

for ax, (method_name, imputed) in zip(axes[1, :], methods.items()):
    # Compare original vs imputed at previously-observed positions
    non_missing_idx = ~missing_mask
    original_vals = prot_matrix[non_missing_idx][:500]
    imputed_vals = imputed[non_missing_idx][:500]
    ax.scatter(original_vals, imputed_vals, alpha=0.3, s=8)
    lo, hi = original_vals.min(), original_vals.max()
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1)
    from scipy.stats import pearsonr
    r, _ = pearsonr(original_vals, imputed_vals)
    ax.set_title(f'{method_name}\nR = {r:.3f}')
    ax.set_xlabel('Original value'); ax.set_ylabel('Imputed value')

plt.tight_layout()
plt.show()
print("\nKNN imputation generally performs best for MAR data.")
print("Min/2 (half-minimum) is preferred for MNAR (missing not at random) in proteomics.")


## 4. Batch Effect Detection and Correction

Batch effects arise from technical variation: different sequencing runs, laboratory days, operators, reagent lots.

### Detection methods
- **PCA**: samples cluster by batch, not biology → clear batch effect
- **PVCA** (Principal Variance Component Analysis): quantifies proportion of variance from batch vs. biology
- **RLE plot** (Relative Log Expression): systematic shifts indicate batch effects

### Correction methods

| Tool | Method | Use case |
|---|---|---|
| **ComBat** | Empirical Bayes | Single covariate batch correction |
| **ComBat-seq** | Negative binomial | RNA-seq count data |
| **Harmony** | Iterative correction | scRNA-seq integration |
| **limma removeBatchEffect** | Linear model | After normalization |
| **MNN** | Mutual nearest neighbors | Single-cell multi-batch |

### Important caveat
**Never correct batch when it is confounded with biology** (e.g., all tumor samples in Batch1, all normals in Batch2). Always check confounding with `table(batch, type)` first.


In [ ]:
# ----- Batch effect detection and correction -----
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_pca_batch(data, title, ax_type, ax_batch, type_colors, batch_colors):
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(data)
    for s in range(len(pcs)):
        ax_type.scatter(pcs[s,0], pcs[s,1], c=type_colors[sample_type[s]], s=20, alpha=0.7)
        ax_batch.scatter(pcs[s,0], pcs[s,1], c=batch_colors[batch[s]], s=20, alpha=0.7)
    var1, var2 = pca.explained_variance_ratio_
    ax_type.set_xlabel(f'PC1 ({var1*100:.1f}%)'); ax_type.set_ylabel(f'PC2 ({var2*100:.1f}%)')
    ax_batch.set_xlabel(f'PC1 ({var1*100:.1f}%)'); ax_batch.set_ylabel(f'PC2 ({var2*100:.1f}%)')

def combat_correct(data, batch_labels):
    """Simplified ComBat-like batch correction (mean-only)."""
    batches = sorted(set(batch_labels))
    corrected = data.copy()
    grand_mean = data.mean(axis=0)
    for b in batches:
        mask = np.array(batch_labels) == b
        batch_mean = data[mask].mean(axis=0)
        corrected[mask] -= (batch_mean - grand_mean)
    return corrected

type_colors = {'Normal': 'green', 'TumorA': 'orange', 'TumorB': 'red'}
batch_colors = {'Batch1': '#3498DB', 'Batch2': '#E74C3C'}

rna_corrected = combat_correct(rna_matrix, batch)
prot_corrected = combat_correct(prot_matrix, batch)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Batch Effect Detection and Correction', fontsize=13, fontweight='bold')

omics_data = [
    ('RNA-seq (raw)', rna_matrix),
    ('RNA-seq (corrected)', rna_corrected),
    ('Proteomics (raw)', prot_matrix),
    ('Proteomics (corrected)', prot_corrected),
]

# Row 0: colored by sample type; Row 1: colored by batch; Row 2: variance explained
for col, (label, data) in enumerate(omics_data):
    # PCA by type
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(data)
    var1, var2 = pca.explained_variance_ratio_[:2]
    
    for s in range(n_samples):
        axes[0, col].scatter(pcs[s,0], pcs[s,1], c=type_colors[sample_type[s]], s=20, alpha=0.7)
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].set_xlabel(f'PC1 ({var1*100:.1f}%)')
    if col == 0: axes[0, col].set_ylabel('Sample type')
    
    for s in range(n_samples):
        axes[1, col].scatter(pcs[s,0], pcs[s,1], c=batch_colors[batch[s]], s=20, alpha=0.7)
    axes[1, col].set_xlabel(f'PC1 ({var1*100:.1f}%)')
    if col == 0: axes[1, col].set_ylabel('Batch')
    
    # Variance explained
    pca_all = PCA(n_components=10)
    pca_all.fit(data)
    axes[2, col].bar(range(10), pca_all.explained_variance_ratio_ * 100, color='steelblue', alpha=0.8)
    axes[2, col].set_xlabel('PC')
    if col == 0: axes[2, col].set_ylabel('Var explained (%)')
    axes[2, col].set_xticks(range(10)); axes[2, col].set_xticklabels([f'PC{i+1}' for i in range(10)], rotation=45, fontsize=7)

# Legends
type_handles = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
batch_handles = [mpatches.Patch(color=v, label=k) for k, v in batch_colors.items()]
axes[0, -1].legend(handles=type_handles, fontsize=8, loc='lower right')
axes[1, -1].legend(handles=batch_handles, fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()
print("After ComBat correction: biological signal in PC1, batch effect removed.")


## 5. Sample Matching and Feature Selection

### Sample matching challenge
Not every sample has every omics layer measured. Integration requires careful handling:
- **Complete case analysis**: use only samples with all omics → loses power
- **Imputation across modalities**: risky, can introduce artifacts
- **MOFA2** handles missing modalities natively by treating them as missing views

### Feature pre-selection
With p >> n, feature selection before integration is critical:
- **Variance filter**: remove bottom 20% lowest variance features
- **Coefficient of variation**: CV = std/mean (relative variance)
- **Differential expression**: use only DE genes from a prior analysis
- **Importance from sparse models**: lasso/elastic net selected features

### Cross-omics correlation check
RNA-protein correlations for matched genes provide a sanity check: expect r ~ 0.3-0.5 for biologically meaningful datasets. Very low correlation suggests technical issues.


In [ ]:
# ----- Sample matching and feature scaling across omics -----

# Simulate scenario: not all samples have all three omics layers
np.random.seed(5)
missing_prot_idx = np.random.choice(range(n_samples), 8, replace=False)  # 8 samples missing proteomics
missing_meth_idx = np.random.choice(range(n_samples), 5, replace=False)  # 5 missing methylation

has_rna = set(range(n_samples))
has_prot = set(range(n_samples)) - set(missing_prot_idx)
has_meth = set(range(n_samples)) - set(missing_meth_idx)

complete_samples = has_rna & has_prot & has_meth
partial_samples = has_rna - complete_samples

print(f"Samples with all 3 omics layers: {len(complete_samples)}")
print(f"RNA-seq only: {len(has_rna - has_prot - has_meth)}")
print(f"RNA + proteomics (no methylation): {len((has_rna & has_prot) - has_meth)}")

# Venn-style diagram
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Multi-Omics Sample Matching and Scaling', fontsize=12, fontweight='bold')

# Panel 1: sample availability matrix
avail_matrix = np.zeros((n_samples, 3))
avail_matrix[:, 0] = 1  # all have RNA
avail_matrix[list(has_prot), 1] = 1
avail_matrix[list(has_meth), 2] = 1

im = axes[0].imshow(avail_matrix.T, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(['RNA-seq', 'Proteomics', 'Methylation'])
axes[0].set_xlabel('Sample index')
axes[0].set_title('Data Availability Matrix')
plt.colorbar(im, ax=axes[0], label='Available (1=yes)')

# Panel 2: feature dimension comparison
omics_dims = {'RNA-seq\n(2000 genes)': n_rna, 'Proteomics\n(300 proteins)': n_prot,
              'Methylation\n(500 CpGs)': n_meth}
bars = axes[1].bar(omics_dims.keys(), omics_dims.values(),
                   color=['steelblue', 'orange', 'green'], alpha=0.8)
axes[1].set_ylabel('Number of features')
axes[1].set_title('Feature Dimensions per Omics Layer')
for bar, val in zip(bars, omics_dims.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Panel 3: correlation between omics for complete samples
from scipy.stats import pearsonr
complete_list = sorted(complete_samples)
rna_sub = rna_matrix[complete_list]
prot_sub = prot_matrix[complete_list]

# Correlated features (RNA vs protein for first 100 shared features)
# Simulate RNA-protein correlation
r_vals = []
for k in range(min(100, n_prot)):
    r, _ = pearsonr(rna_sub[:, k], prot_sub[:, k])
    r_vals.append(r)

axes[2].hist(r_vals, bins=20, color='purple', alpha=0.8)
axes[2].axvline(0, color='black', linestyle='--')
axes[2].axvline(np.mean(r_vals), color='red', linestyle='--',
                label=f'Mean r = {np.mean(r_vals):.2f}')
axes[2].set_xlabel('Pearson r (RNA vs Protein)')
axes[2].set_ylabel('Count')
axes[2].set_title('RNA-Protein Correlation\n(matched features)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nComplete cases (all 3 omics): {len(complete_samples)}/{n_samples}")
print(f"Mean RNA-protein correlation: {np.mean(r_vals):.3f}")
print("Note: RNA-protein correlations are typically weak (r~0.3-0.4) due to post-translational regulation.")


## 6. Summary

### Key concepts covered

1. **Data landscape**: six major omics layers with distinct properties; integration strategy depends on question (unsupervised exploration vs. supervised classification)
2. **Normalization**: each layer needs appropriate normalization; z-score within-layer is the standard for integration
3. **Missing data**: MNAR predominates in proteomics; min/2 for detection-limit missingness; KNN for MAR
4. **Batch effects**: PCA is the first diagnostic tool; ComBat for correction; never correct when confounded with biology
5. **Sample matching**: MOFA2 handles missing views; otherwise complete-case analysis is safest

### Harmonization checklist
- [ ] Each omics layer normalized independently (TMM/VST for RNA; median centering for proteomics)
- [ ] Methylation beta-values transformed to M-values for modeling
- [ ] Missing rate per feature computed; features >50% missing removed
- [ ] Imputation strategy documented (KNN vs. min/2)
- [ ] Batch effects detected with PCA; correction applied if batch != biology
- [ ] Sample IDs matched across layers (watch for ID swaps!)
- [ ] Feature dimensions reduced with variance filter

### Next steps
- [Notebook 2: MOFA2](02_mofa2.ipynb) — unsupervised factor analysis for multi-omics
- [Notebook 3: mixOmics](03_mixomics.ipynb) — supervised integration with DIABLO


---[< Previous: QIIME2 16S Amplicon Workflow](../26_Metagenomics_Shotgun/04_qiime2_16s.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: MOFA2: Multi-Omics Factor Analysis >](02_mofa2.ipynb)---